## v0.1

In [1]:
INPUT_FILE = "../data/spotify.text2"
CANIDATES_OUTFILE = "../output/data/spotify_candidates_v0.1.tsv"
IDS_OUTFILE = "../output/lists/initial_ids_v0.1.lst"

SAMPLE_SIZE = 7_000 # Random sample of IDs
SEED = 33

### Run

In [2]:
import pandas as pd
df = pd.read_csv(INPUT_FILE, sep="\t", names=["podcast_id", "text"]) 

In [3]:
# We will filter very short and very long podcasts:
df["length"] = df["text"].str.count(" ") + 1

In [4]:
print(df["length"].describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]))

count    107404.000000
mean       6009.961510
std        4368.489162
min           7.000000
1%          159.000000
5%          429.000000
10%         758.000000
25%        2131.000000
50%        5433.000000
75%        9115.000000
90%       12233.700000
95%       13965.850000
99%       16751.940000
max       44425.000000
Name: length, dtype: float64


In [5]:
print((df["length"] < 800).sum() / len(df) * 100)
print((df["length"] > 12_000).sum() / len(df) * 100)

10.52474768165059
10.86737924099661


In [6]:
df_candidates = df.copy()

In [7]:
### Exclude based on length:
min_length = 800
max_length = 12_000
mask = (df["length"] < min_length) | (df["length"] > max_length)
df_candidates = df_candidates[~mask]

print(f"Excluded {mask.sum()} rows based on length")

Excluded 22976 rows based on length


In [8]:
### Exclude manually based on lang:
filter_words = {
    "hindi": [
        'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ए', 'ऐ', 'ओ', 'औ',  
        'क', 'ख', 'ग', 'घ', 'च', 'छ', 'ज', 'झ', 'ट', 'ठ',  
        'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ',  
        'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स',  
        'ह', 'क्ष', 'त्र', 'ज्ञ',  
        '१', '२', '३', '४', '५', '६', '७', '८', '९',  
        'ा', 'ि', 'ी', 'ु', 'ू', 'े', 'ै', 'ो', 'ौ', 'ं', '्'
    ],
    "chinese": [
        '的', '是', '了', '我', '不', '人', '在', '他', '有',
        '这', '中', '大', '来', '上', '国', '个', '到', '说', '们',
        '和', '地', '也', '子', '时', '道', '出', '而', '要', '于'
    ],
    "japanese": [
        'あ', 'い', 'う', 'え', 'お', 'か', 'き', 'く', 'け', 'こ',
        'さ', 'し', 'す', 'せ', 'そ', 'た', 'ち', 'つ', 'て', 'と',
        'な', 'に', 'ぬ', 'ね', 'の', 'は', 'ひ', 'ふ', 'へ', 'ほ',
        'ま', 'み', 'む', 'め', 'も', 'や', 'ゆ', 'よ', 'ら', 'り',
        'る', 'れ', 'ろ', 'わ', 'を', 'ん',
        'ア', 'イ', 'ウ', 'エ', 'オ', 'カ', 'キ', 'ク', 'ケ', 'コ',
        'サ', 'シ', 'ス', 'セ', 'ソ', 'タ', 'チ', 'ツ', 'テ', 'ト',
        'ナ', 'ニ', 'ヌ', 'ネ', 'ハ', 'ヒ', 'フ', 'ヘ', 'ホ',
        'マ', 'ミ', 'ム', 'メ', 'モ', 'ヤ', 'ユ', 'ヨ', 'ラ', 'リ',
        'ル', 'レ', 'ロ', 'ワ', 'ヲ', 'ン'
    ],
    "arabic": [
        'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر',
        'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف',
        'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ي', 'ء', 'ى',
        'ة', '٢', '٣', '٤', '٥', '٦', '٧', '٨', '٩', '٠'
    ],
    "russian": [
        'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ё', 'Ж', 'З', 'И',
        'Й', 'К', 'Л', 'М', 'Н', 'О', 'П', 'Р', 'С', 'Т',
        'У', 'Ф', 'Х', 'Ц', 'Ч', 'Ш', 'Щ', 'Ъ', 'Ы', 'Ь',
        'Э', 'Ю', 'Я', 'а', 'б', 'в', 'г', 'д', 'е', 'ё',
        'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п',
        'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ',
        'ъ', 'ы', 'ь', 'э', 'ю', 'я'
    ],
    "korean": [
        '가', '나', '다', '라', '마', '바', '사', '아', '자', '차',
        '카', '타', '파', '하', '것', '들', '한', '그', '너', '저',
        '우리', '너희', '안', '좋', '말', '생각', '하', '보', '오',
        '이', '는', '를', '과', '에', '들', '이다', '입니다'
    ],

}

masks = {}

for lang, words in filter_words.items():
    masks[lang] = df_candidates["text"].str.contains("|".join(words), case=False, regex=True)
    print(f"Excluded {masks[lang].sum()} rows based on {lang}")

Excluded 0 rows based on hindi
Excluded 52 rows based on chinese
Excluded 24 rows based on japanese
Excluded 239 rows based on arabic
Excluded 34 rows based on russian
Excluded 81 rows based on korean


In [9]:
full_mask = pd.concat(masks.values(), axis=1).any(axis=1)
print(f"Excluded {full_mask.sum()} rows based on language (manual)")

Excluded 420 rows based on language (manual)


In [10]:
df_candidates = df_candidates[~full_mask]

In [11]:
### Exclude based on language detection (fasttext):
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")
model = fasttext.load_model(model_path)

In [12]:
import numpy as np

def detect_language(texts: pd.Series, model, max_chars=500):
    texts = texts.str.slice(0, max_chars)
    results = model.predict(texts.tolist())
    labels = [r[0].split("__label__")[1] for r in results[0]]
    probas = [r for r in results[1]]
    probas = np.array(probas).flatten()
    return labels, probas

labels, probas = detect_language(df_candidates["text"], model)

In [13]:
df_candidates["lang_ft"] = labels
df_candidates["lang_proba_ft"] = probas

In [14]:
mask_keep = (df_candidates["lang_ft"] == "eng_Latn") & (df_candidates["lang_proba_ft"] > 0.9)
df_candidates = df_candidates[mask_keep]
print(f"Excluded {(~mask_keep).sum()} rows based on language detection (fasttext)")

Excluded 458 rows based on language detection (fasttext)


In [15]:
print(df_candidates.shape)

(83550, 5)


In [16]:
# Save the candidates as tsv with cols id, text:
from pathlib import Path

Path(CANIDATES_OUTFILE).parent.mkdir(parents=True, exist_ok=True)

df_candidates[["podcast_id", "text"]].to_csv(CANIDATES_OUTFILE, sep="\t", index=False)

In [18]:
df_sample = df_candidates.sample(SAMPLE_SIZE, random_state=SEED)

# Save the IDs to a file of IDs sep by newlines:
Path(IDS_OUTFILE).parent.mkdir(parents=True, exist_ok=True)
with open(IDS_OUTFILE, "w") as f:
    for id_ in df_sample["podcast_id"]:
        f.write(f"{id_}\n")

------------------